# Custom Loss Functions with LoRA

## Historical Context: Loss Functions Meet Parameter-Efficient Training

### The Evolution

**Stage 1: Full Fine-Tuning with Custom Losses**
- Weighted Cross-Entropy for class imbalance
- Focal loss for hard examples
- Custom multi-task objectives
- **Problem:** Requires full model training (expensive)

**Stage 2: LoRA Introduction (2021)**
- Efficient parameter updates
- Initially used standard cross-entropy
- **Question:** Do custom losses work with frozen base models?

**Stage 3: Integration (2022-2024)**
- Discovery: ALL custom losses work with LoRA!
- Gradient flows correctly through LoRA adapters
- Can combine PEFT + advanced loss functions

### Key Insight

**LoRA doesn't restrict loss functions:**
```
Forward Pass:
Input → Frozen Base Model + LoRA Adapters → Output → Custom Loss

Backward Pass:
Custom Loss Gradient → LoRA Adapters (trainable) ← Gradients flow here
                     → Base Model (frozen) ← No gradients needed
```

The loss function only cares about outputs, not which parameters are trainable!

## Problem Statement

Apply custom loss functions from Stage 1 to LoRA/QLoRA training:
1. Weighted Cross-Entropy with class imbalance
2. Focal loss for hard examples
3. Multi-task learning with multiple adapters
4. Verify gradient flow
5. Debug training issues

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Optional, Tuple
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1: Weighted Cross-Entropy with LoRA

### Scenario: Imbalanced Classification

Recall from Notebook 14:
- Class 0 (negative): 9000 examples
- Class 1 (positive): 1000 examples

**Solution:** Weighted CE to balance class importance

```python
# Calculate class weights
class_counts = [9000, 1000]
total = sum(class_counts)
weights = [total / (len(class_counts) * count) for count in class_counts]
# weights = [0.56, 5.0]
```

### Implementation with LoRA

The key: Custom loss in Trainer's `compute_loss` method

In [ ]:
# Load model and add LoRA
model_id = "distilbert-base-uncased"

print("Loading base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    torch_dtype=torch.float32
)

print("\nAdding LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention modules
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("\nModel ready for weighted loss training!")

In [ ]:
# Custom Trainer with Weighted CE Loss
class WeightedCELoRATrainer(Trainer):
    """
    Custom Trainer that applies weighted cross-entropy loss with LoRA.
    
    The loss is computed on the outputs from the LoRA-enhanced model.
    Gradients flow back through LoRA adapters (trainable) but not base model (frozen).
    """
    
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
        if class_weights is not None:
            # Move weights to same device as model
            self.class_weights = torch.FloatTensor(class_weights)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Compute weighted cross-entropy loss.
        
        Args:
            model: LoRA-enhanced model
            inputs: Batch of inputs
            return_outputs: Whether to return model outputs
        
        Returns:
            Loss (and optionally outputs)
        """
        # Extract labels
        labels = inputs.pop("labels")
        
        # Forward pass through LoRA model
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Move class weights to correct device
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
        else:
            weights = None
        
        # Compute weighted cross-entropy
        loss = F.cross_entropy(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1),
            weight=weights
        )
        
        return (loss, outputs) if return_outputs else loss

print("=== Weighted CE LoRA Trainer ===" )
print("\nHow it works:")
print("1. Forward: Input → Frozen Base + LoRA → Logits")
print("2. Loss: Weighted CE on logits")
print("3. Backward: Gradients flow only to LoRA adapters")
print("4. Update: Only LoRA parameters updated")
print("\nBase model remains frozen throughout!")

In [ ]:
# Demonstrate gradient flow
print("=== Verifying Gradient Flow ===\n")

# Create dummy batch
dummy_input = {
    "input_ids": torch.randint(0, 1000, (4, 128)),
    "attention_mask": torch.ones((4, 128)),
    "labels": torch.tensor([0, 1, 0, 1])
}

# Calculate class weights
class_weights = [0.56, 5.0]  # Favor minority class

# Forward pass
model.train()
outputs = model(
    input_ids=dummy_input["input_ids"],
    attention_mask=dummy_input["attention_mask"]
)
logits = outputs.logits

# Compute weighted loss
weights = torch.FloatTensor(class_weights)
loss = F.cross_entropy(
    logits,
    dummy_input["labels"],
    weight=weights
)

print(f"Loss value: {loss.item():.4f}")
print(f"\nBefore backward pass:")
print(f"  Logits require grad: {logits.requires_grad}")

# Backward pass
loss.backward()

# Check gradients
print(f"\nAfter backward pass:")
lora_params_with_grad = 0
base_params_with_grad = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        if param.grad is not None:
            if "lora" in name.lower():
                lora_params_with_grad += 1
            else:
                base_params_with_grad += 1

print(f"  LoRA params with gradients: {lora_params_with_grad}")
print(f"  Base params with gradients: {base_params_with_grad}")
print("\n✓ Gradients flow correctly to LoRA only!")

## Step 2: Focal Loss with LoRA

### Scenario: Hard Example Mining

From Notebook 14, recall Focal Loss:
```python
FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)

where:
  p_t = model's predicted probability for true class
  γ = focusing parameter (typically 2.0)
  α_t = class weighting factor
```

**Benefits with LoRA:**
- Focus on hard examples during fine-tuning
- Prevents overfitting to easy examples
- Works seamlessly with frozen base models

In [ ]:
# Focal Loss implementation
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance and hard examples.
    
    Compatible with LoRA and any PEFT method.
    """
    
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        Args:
            alpha: Class weights (list or tensor)
            gamma: Focusing parameter (0 = CE, higher = more focus on hard)
            reduction: 'mean', 'sum', or 'none'
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Logits from model (batch_size, num_classes)
            targets: Ground truth labels (batch_size,)
        
        Returns:
            Focal loss value
        """
        # Convert logits to probabilities
        p = F.softmax(inputs, dim=1)
        
        # Get probability of true class
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = p.gather(1, targets.view(-1, 1)).squeeze(1)
        
        # Focal term: (1 - p_t)^gamma
        focal_term = (1 - p_t) ** self.gamma
        
        # Apply focal term
        loss = focal_term * ce_loss
        
        # Apply class weights if provided
        if self.alpha is not None:
            if isinstance(self.alpha, (list, np.ndarray)):
                alpha = torch.FloatTensor(self.alpha).to(inputs.device)
            else:
                alpha = self.alpha.to(inputs.device)
            alpha_t = alpha.gather(0, targets)
            loss = alpha_t * loss
        
        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# Test focal loss
print("=== Focal Loss with LoRA ===\n")

focal_loss_fn = FocalLoss(alpha=[0.56, 5.0], gamma=2.0)

# Simulate predictions
logits = torch.tensor([
    [2.0, -1.0],  # Easy negative (high confidence)
    [-1.0, 2.0],  # Easy positive (high confidence)
    [0.5, 0.4],   # Hard negative (low confidence)
    [-0.4, 0.5]   # Hard positive (low confidence)
])
labels = torch.tensor([0, 1, 0, 1])

# Compare CE vs Focal Loss
ce_loss = F.cross_entropy(logits, labels, reduction='none')
focal_loss = focal_loss_fn(logits, labels)

print("Per-example loss comparison:")
print(f"{'Example':<15} {'CE Loss':<12} {'Focal Loss':<12} {'Ratio':<10}")
print("-" * 50)

focal_losses_per_example = []
for i in range(len(labels)):
    ce_val = ce_loss[i].item()
    
    # Compute focal loss for this example
    p = F.softmax(logits[i:i+1], dim=1)
    p_t = p[0, labels[i]]
    focal_term = (1 - p_t) ** 2.0
    focal_val = focal_term * ce_val
    if labels[i] == 1:
        focal_val = focal_val * 5.0  # Apply alpha
    else:
        focal_val = focal_val * 0.56
    focal_losses_per_example.append(focal_val.item())
    
    ratio = focal_val.item() / ce_val if ce_val > 0 else 0
    example_type = "Easy" if i < 2 else "Hard"
    print(f"{example_type} ex {i+1:<7} {ce_val:<12.4f} {focal_val.item():<12.4f} {ratio:<10.2f}")

print("\nObservation:")
print("  - Hard examples have higher loss (ratio > 1)")
print("  - Easy examples have lower loss (ratio < 1)")
print("  - Model focuses learning on hard cases")

In [ ]:
# Custom Trainer with Focal Loss
class FocalLossLoRATrainer(Trainer):
    """Trainer with Focal Loss for LoRA models."""
    
    def __init__(self, *args, focal_alpha=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Compute focal loss
        loss = self.focal_loss(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("=== Focal Loss LoRA Trainer ===")
print("\nUsage:")
print("trainer = FocalLossLoRATrainer(")
print("    model=lora_model,")
print("    focal_alpha=[0.56, 5.0],  # Class weights")
print("    focal_gamma=2.0,           # Focusing parameter")
print("    ...)")
print("\nWorks seamlessly with LoRA/QLoRA!")

## Step 3: Multi-Task Learning with Multiple LoRA Adapters

### Scenario: One Model, Multiple Tasks

**Approach: Task-Specific LoRA Adapters**

```
Base Model (Frozen)
    ├── LoRA Adapter A (Sentiment Analysis)
    ├── LoRA Adapter B (Topic Classification)
    └── LoRA Adapter C (Named Entity Recognition)
```

**Training Strategy:**
1. Load base model once
2. Add task-specific LoRA adapters
3. Train with multi-task loss
4. Each task has its own loss weight

### Multi-Task Loss

```python
total_loss = λ1 * loss_task1 + λ2 * loss_task2 + λ3 * loss_task3

where λi are task weights (hyperparameters)
```

In [ ]:
# Multi-task LoRA setup (conceptual demonstration)
print("=== Multi-Task Learning with LoRA ===\n")

# Conceptual: Multiple task-specific adapters
print("Setup:")
print("1. Load base model")
print("2. Create separate LoRA configs for each task:\n")

task_configs = {
    "sentiment": {
        "r": 16,
        "target_modules": ["q_lin", "v_lin"],
        "task_type": TaskType.SEQ_CLS,
        "num_labels": 3  # positive/negative/neutral
    },
    "topic": {
        "r": 16,
        "target_modules": ["q_lin", "v_lin"],
        "task_type": TaskType.SEQ_CLS,
        "num_labels": 10  # 10 topic categories
    },
    "ner": {
        "r": 16,
        "target_modules": ["q_lin", "v_lin"],
        "task_type": TaskType.TOKEN_CLS,
        "num_labels": 9  # NER tags
    }
}

for task_name, config in task_configs.items():
    print(f"   Task: {task_name}")
    print(f"     - Rank: {config['r']}")
    print(f"     - Labels: {config['num_labels']}")
    print()

print("3. During training:")
print("   - Batch contains examples from all tasks")
print("   - Compute loss for each task")
print("   - Weighted sum: total_loss = Σ λi * loss_i")
print("   - Backprop through all active adapters")
print("\n4. Inference:")
print("   - Load base model + specific adapter")
print("   - Fast task switching (just swap adapter)")

In [ ]:
# Multi-task loss implementation
class MultiTaskLoss(nn.Module):
    """
    Multi-task loss for training multiple LoRA adapters jointly.
    """
    
    def __init__(self, task_weights: Dict[str, float]):
        """
        Args:
            task_weights: Dictionary mapping task names to weights
                         e.g., {'sentiment': 1.0, 'topic': 0.5, 'ner': 2.0}
        """
        super().__init__()
        self.task_weights = task_weights
    
    def forward(self, task_losses: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        Compute weighted sum of task losses.
        
        Args:
            task_losses: Dictionary of task-specific losses
        
        Returns:
            Combined weighted loss
        """
        total_loss = 0.0
        
        for task_name, loss in task_losses.items():
            weight = self.task_weights.get(task_name, 1.0)
            total_loss += weight * loss
        
        return total_loss

# Example usage
print("=== Multi-Task Loss Example ===\n")

# Define task weights
task_weights = {
    "sentiment": 1.0,   # Standard weight
    "topic": 0.5,       # Less important
    "ner": 2.0         # More important (harder task)
}

multi_task_loss = MultiTaskLoss(task_weights)

# Simulate task-specific losses
task_losses = {
    "sentiment": torch.tensor(0.8),
    "topic": torch.tensor(1.2),
    "ner": torch.tensor(1.5)
}

# Compute combined loss
total_loss = multi_task_loss(task_losses)

print("Task-specific losses:")
for task, loss in task_losses.items():
    weight = task_weights[task]
    weighted = weight * loss.item()
    print(f"  {task}: {loss.item():.4f} × {weight} = {weighted:.4f}")

print(f"\nTotal weighted loss: {total_loss.item():.4f}")
print("\nThis loss is backpropagated through all LoRA adapters simultaneously!")

## Step 4: Debugging LoRA Training

### Common Issues and Solutions

**Issue 1: Loss Not Decreasing**
- Check: Are LoRA parameters trainable?
- Check: Is learning rate appropriate (higher than full FT)?
- Check: Are gradients flowing correctly?

**Issue 2: Poor Quality Despite Low Loss**
- Check: Is rank too low (r < 8)?
- Check: Are target modules appropriate for task?
- Check: Is base model too small?

**Issue 3: Memory Issues**
- Check: Gradient accumulation enabled?
- Check: Using FP16/BF16?
- Check: Too many target modules?

### Debugging Tools

In [ ]:
# Debugging utilities
def diagnose_lora_model(model, verbose=True):
    """
    Diagnose LoRA model configuration and training state.
    
    Args:
        model: PEFT model with LoRA adapters
        verbose: Print detailed information
    
    Returns:
        Dictionary with diagnostic information
    """
    diagnostics = {
        "total_params": 0,
        "trainable_params": 0,
        "lora_params": 0,
        "frozen_params": 0,
        "has_gradients": False,
        "lora_modules": []
    }
    
    for name, param in model.named_parameters():
        diagnostics["total_params"] += param.numel()
        
        if param.requires_grad:
            diagnostics["trainable_params"] += param.numel()
            if "lora" in name.lower():
                diagnostics["lora_params"] += param.numel()
                diagnostics["lora_modules"].append(name)
            
            if param.grad is not None:
                diagnostics["has_gradients"] = True
        else:
            diagnostics["frozen_params"] += param.numel()
    
    if verbose:
        print("=== LoRA Model Diagnostics ===\n")
        print(f"Total parameters: {diagnostics['total_params']:,}")
        print(f"Trainable parameters: {diagnostics['trainable_params']:,} "
              f"({100 * diagnostics['trainable_params'] / diagnostics['total_params']:.4f}%)")
        print(f"  ├── LoRA parameters: {diagnostics['lora_params']:,}")
        print(f"  └── Other trainable: {diagnostics['trainable_params'] - diagnostics['lora_params']:,}")
        print(f"Frozen parameters: {diagnostics['frozen_params']:,}")
        print(f"\nGradients computed: {diagnostics['has_gradients']}")
        print(f"\nLoRA modules ({len(diagnostics['lora_modules'])}):")
        for module_name in diagnostics['lora_modules'][:5]:  # Show first 5
            print(f"  - {module_name}")
        if len(diagnostics['lora_modules']) > 5:
            print(f"  ... and {len(diagnostics['lora_modules']) - 5} more")
    
    return diagnostics

# Run diagnostics on our model
diagnostics = diagnose_lora_model(model)

# Verify configuration
print("\n=== Configuration Check ===")
checks = {
    "LoRA params exist": diagnostics["lora_params"] > 0,
    "Base model frozen": diagnostics["frozen_params"] > 0,
    "Trainable ratio < 10%": (diagnostics["trainable_params"] / diagnostics["total_params"]) < 0.1,
}

for check_name, passed in checks.items():
    status = "✓" if passed else "✗"
    print(f"{status} {check_name}")

In [ ]:
# Gradient flow verification
def verify_gradient_flow(model, sample_input):
    """
    Verify that gradients flow correctly through LoRA adapters.
    
    Args:
        model: PEFT model
        sample_input: Dictionary with input_ids, attention_mask, labels
    
    Returns:
        Dictionary with gradient flow information
    """
    model.train()
    
    # Forward pass
    outputs = model(
        input_ids=sample_input["input_ids"],
        attention_mask=sample_input["attention_mask"],
        labels=sample_input["labels"]
    )
    loss = outputs.loss
    
    # Backward pass
    loss.backward()
    
    # Check gradient flow
    grad_info = {
        "loss_value": loss.item(),
        "lora_grads": [],
        "base_grads": [],
        "lora_grad_norms": [],
        "base_grad_norms": []
    }
    
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            
            if "lora" in name.lower():
                grad_info["lora_grads"].append(name)
                grad_info["lora_grad_norms"].append(grad_norm)
            else:
                grad_info["base_grads"].append(name)
                grad_info["base_grad_norms"].append(grad_norm)
    
    print("=== Gradient Flow Verification ===\n")
    print(f"Loss: {grad_info['loss_value']:.4f}")
    print(f"\nLoRA parameters with gradients: {len(grad_info['lora_grads'])}")
    print(f"Base parameters with gradients: {len(grad_info['base_grads'])}")
    
    if grad_info["lora_grad_norms"]:
        print(f"\nLoRA gradient statistics:")
        print(f"  Mean norm: {np.mean(grad_info['lora_grad_norms']):.6f}")
        print(f"  Max norm: {np.max(grad_info['lora_grad_norms']):.6f}")
        print(f"  Min norm: {np.min(grad_info['lora_grad_norms']):.6f}")
    
    # Verdict
    print("\n=== Verdict ===")
    if len(grad_info["lora_grads"]) > 0 and len(grad_info["base_grads"]) == 0:
        print("✓ Correct: Gradients flow only to LoRA parameters")
    elif len(grad_info["lora_grads"]) == 0:
        print("✗ Problem: No gradients in LoRA parameters!")
        print("  → Check if LoRA layers are properly registered")
    elif len(grad_info["base_grads"]) > 0:
        print("⚠ Warning: Gradients flowing to base model")
        print("  → Check if base model was properly frozen")
    
    return grad_info

# Test gradient flow
test_input = {
    "input_ids": torch.randint(0, 1000, (2, 64)),
    "attention_mask": torch.ones((2, 64)),
    "labels": torch.tensor([0, 1])
}

grad_info = verify_gradient_flow(model, test_input)

## Step 5: Practical Examples

### Example 1: Imbalanced Sentiment Analysis with LoRA

In [ ]:
# Complete example: Weighted CE with LoRA
print("=== Complete Example: Sentiment Analysis with Class Imbalance ===\n")

# Simulate imbalanced dataset
print("Dataset:")
print("  Negative examples: 9000")
print("  Positive examples: 1000")
print("  Neutral examples: 500")
print("  Imbalance ratio: 18:1")

# Calculate class weights
class_counts = [9000, 1000, 500]
num_classes = len(class_counts)
total_samples = sum(class_counts)
class_weights = [total_samples / (num_classes * count) for count in class_counts]

print(f"\nClass weights: {[f'{w:.2f}' for w in class_weights]}")
print("  → Negative: 0.37 (downweight majority)")
print("  → Positive: 3.33 (upweight minority)")
print("  → Neutral: 6.67 (upweight smallest)")

# Create LoRA config for sentiment analysis
lora_config_sentiment = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],  # Attention only sufficient for classification
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

print("\nLoRA Configuration:")
print(f"  Rank: {lora_config_sentiment.r}")
print(f"  Target: {lora_config_sentiment.target_modules}")
print(f"  Dropout: {lora_config_sentiment.lora_dropout}")

print("\nTraining Setup:")
print("  1. Load DistilBERT base")
print("  2. Add LoRA adapters")
print("  3. Use WeightedCELoRATrainer")
print("  4. Train only LoRA params (~300K vs 67M full model)")
print("\nExpected:")
print("  - Better minority class recall")
print("  - Balanced F1 scores across classes")
print("  - 200x fewer trainable parameters")

### Example 2: Hard Example Mining with Focal Loss

In [ ]:
# Example: Focal loss for difficult classification
print("=== Example: Focal Loss for Hard Examples ===\n")

print("Scenario:")
print("  - Medical diagnosis task")
print("  - Some conditions are subtle (hard to classify)")
print("  - Easy cases dominate the dataset")
print("  - Need model to focus on hard cases")

print("\nSolution: Focal Loss + LoRA")
print("  - Focal Loss: Downweight easy examples")
print("  - LoRA: Efficient fine-tuning")

# Focal loss config
focal_alpha = [1.0, 2.0, 3.0]  # Weight harder-to-diagnose conditions more
focal_gamma = 2.5  # Strong focusing (higher than default 2.0)

print(f"\nFocal Loss Configuration:")
print(f"  Alpha (class weights): {focal_alpha}")
print(f"  Gamma (focusing): {focal_gamma}")
print("\nEffect:")
print("  - Easy examples (p > 0.9): Loss reduced by ~100x")
print("  - Hard examples (p < 0.5): Loss almost unchanged")
print("  - Result: Model allocates gradient budget to hard cases")

# Training setup
print("\nTraining:")
print("  trainer = FocalLossLoRATrainer(")
print("      model=lora_model,")
print(f"      focal_alpha={focal_alpha},")
print(f"      focal_gamma={focal_gamma},")
print("      args=training_args")
print("  )")
print("\n  trainer.train()")

print("\nBenefits:")
print("  ✓ Better performance on rare/difficult cases")
print("  ✓ Prevents overfitting to easy majority")
print("  ✓ Works seamlessly with frozen base model")
print("  ✓ Only ~0.5% parameters trainable")

## Summary: Custom Loss Functions with LoRA

### What We Learned

1. **All Loss Functions Work with LoRA**
   - Any loss that works with full fine-tuning works with LoRA
   - Gradients flow correctly to trainable adapters
   - Base model remains frozen

2. **Implementation Pattern**
   - Override `compute_loss` in Trainer
   - Compute custom loss on model outputs
   - Backward pass handles gradient routing automatically

3. **Common Use Cases**
   - Weighted CE: Class imbalance
   - Focal Loss: Hard example mining
   - Multi-task: Multiple task-specific adapters
   - Custom: Domain-specific objectives

### Quick Reference

**Weighted Cross-Entropy:**
```python
class WeightedCELoRATrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.FloatTensor(class_weights)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(
            outputs.logits, labels, 
            weight=self.class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss
```

**Focal Loss:**
```python
class FocalLossLoRATrainer(Trainer):
    def __init__(self, *args, focal_alpha=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss
```

**Multi-Task:**
```python
# Compute losses for each task
task_losses = {
    "task1": loss1,
    "task2": loss2,
    "task3": loss3
}

# Weighted combination
total_loss = sum(weight * loss for task, loss in task_losses.items())
```

### Debugging Checklist

When custom loss + LoRA isn't working:

1. **Check Trainability**
   - Run `model.print_trainable_parameters()`
   - Verify LoRA params are trainable
   - Verify base model is frozen

2. **Check Gradients**
   - Run `verify_gradient_flow(model, sample)`
   - Ensure gradients only in LoRA layers
   - Check gradient magnitudes aren't zero

3. **Check Loss**
   - Print loss values during training
   - Verify loss decreases over time
   - Compare with standard CE baseline

4. **Check Configuration**
   - Rank not too low (r >= 8)
   - Learning rate higher than full FT (2e-4 vs 2e-5)
   - Target modules appropriate for task

### Best Practices

1. **Start Simple**
   - Test with standard CE first
   - Add custom loss once baseline works
   - Debug incrementally

2. **Loss Scaling**
   - Custom losses may have different magnitudes
   - Monitor absolute loss values
   - Adjust learning rate if needed

3. **Validation**
   - Always validate on held-out data
   - Custom losses can overfit differently
   - Use early stopping

4. **Documentation**
   - Document loss hyperparameters
   - Record class weights calculation
   - Note any task-specific tuning

### Combining Everything

**Ultimate Setup: QLoRA + Focal Loss + Multi-Task**
```python
# 1. Load base model in 4-bit
quantization_config = BitsAndBytesConfig(load_in_4bit=True, ...)
base_model = AutoModelForCausalLM.from_pretrained(
    "llama-2-7b",
    quantization_config=quantization_config
)

# 2. Add LoRA adapters
lora_config = LoraConfig(r=16, ...)
model = get_peft_model(base_model, lora_config)

# 3. Custom multi-task focal loss trainer
trainer = MultiTaskFocalLossTrainer(
    model=model,
    task_weights={"task1": 1.0, "task2": 2.0},
    focal_gamma=2.0,
    ...
)

# 4. Train
trainer.train()

# Result:
# - 4-bit quantization: 4x memory savings
# - LoRA: 100-1000x fewer parameters
# - Focal loss: Better hard example performance
# - Multi-task: One model for multiple tasks
# - Total: State-of-the-art efficiency + quality
```

### References

**Loss Functions:**
- Cross-Entropy: Standard PyTorch
- Focal Loss: Lin et al. (2017) "Focal Loss for Dense Object Detection"
- Weighted CE: Standard imbalanced learning technique

**LoRA:**
- Hu et al. (2021): "LoRA: Low-Rank Adaptation of Large Language Models"
- PEFT Library: https://github.com/huggingface/peft

**Integration:**
- All Stage 1 techniques (Notebooks 13-19) work with LoRA
- Gradient flow is automatic
- No special considerations needed

### Next Steps

You now have complete Stage 2 knowledge:
- **Notebook 21**: LoRA basics with Llama 2
- **Notebook 22**: QLoRA for extreme memory efficiency
- **Notebook 23**: Target module selection strategies
- **Notebook 24**: Custom losses with LoRA (this notebook)

**Stage 3 Preview:**
- Advanced PEFT methods (Adapter layers, Prompt Tuning)
- Multi-adapter strategies
- Production deployment patterns
- Inference optimization